In [1]:
# Test script 2

In [2]:
# Option 2 for mortality equation (population weight X-TMREL) TEST FOR 2000

In [1]:
import os
import xarray as xr
import regionmask
import matplotlib.pyplot as plt
import numpy as np

In [2]:
BMR_DIR = "/glade/work/awells/air_quality/BMR/"
POP_DIR = "/glade/work/awells/air_quality/SSP_pop/SSP2/"
O3_DIR = "/glade/work/awells/air_quality/O3_obs/"

In [ ]:
bmr = xr.open_dataarray(f"{BMR_DIR}GBD_BMR_Country_Mask_COPD_1990-2009.nc")
pop_2000 = xr.open_dataarray(f"{POP_DIR}ssp2_total_regrid_2000.nc")
o3_2000 = xr.open_dataset(f"{O3_DIR}Delang_BME_OSDMA8_1990_2017.nc")["ozone"].sel(year=slice("1990", "2009")).mean("year")

In [ ]:
pop_lat = pop_2000.lat.values
pop_lon = pop_2000.lon.values

o3_lat = o3_2000.latitude.values
o3_lon = o3_2000.longitude.values

bmr_lat = bmr.lat.values
bmr_lon = bmr.lon.values

In [ ]:
o3_lat_mask = xr.ufuncs.logical_and(o3_lat <= pop_lat.max(), o3_lat >= pop_lat.min())
o3_lon_mask = xr.ufuncs.logical_and(o3_lon <= pop_lon.max(), o3_lon >= pop_lon.min())

bmr_lat_mask = xr.ufuncs.logical_and(bmr_lat <= pop_lat.max(), bmr_lat >= pop_lat.min())
bmr_lon_mask = xr.ufuncs.logical_and(bmr_lon <= pop_lon.max(), bmr_lon >= pop_lon.min())

In [ ]:
new_o3 = o3_2000.sel(latitude=o3_lat_mask)
new_bmr = bmr.sel(lat=bmr_lat_mask)

In [ ]:
o3_interp = new_o3.interp(latitude=pop_2000.lat, longitude=pop_2000.lon)
bmr_interp = new_bmr.interp(lat=pop_2000.lat, lon=pop_2000.lon)

In [ ]:
RR_10ppb = 1.074
# RR = e^(beta*(x-TMREL))
beta = np.log(RR_10ppb)/10
TMREL = 32.4

In [ ]:
O3_diff = o3_interp - TMREL
TMREL_O3 = xr.where(O3_diff > 0, O3_diff, 0)

In [ ]:
pop_tot = pop_2000.sum()

In [ ]:
O3_pop = TMREL_O3 * (pop_2000/pop_tot)

In [ ]:
RR = np.exp(beta * O3_pop)

In [ ]:
AF = (RR - 1)/RR

In [ ]:
M = AF * bmr_interp * pop_2000

In [ ]:
# === Latitude weighting sum ===
def weighted_sum(da):
    weights = np.cos(np.deg2rad(da.lat))
    weights.name = "weights"
    new_da = da.weighted(weights).sum(("lon", "lat"))
    return new_da

In [ ]:
new_M = weighted_sum(M)

In [ ]:
new_M

In [ ]:
levels = [0, 50, 100, 500, 1000, 5000, 10000, 50000, 100000]

In [ ]:
M.sel(quantile="median").plot(cmap="Reds", levels=levels)